In [1]:

import wandb
import numpy as np
import joblib
from SRC.helper_functions.preprocessing import processor, prep_x_for_tf_horiz, prep_y_for_tf_horiz
import polars as pl

In [2]:
'''list of all sites    "sites": [
        "06936530", "06930000", "06900050", "06925250", "06929900",
        "06897500", "06901250", "06893970", "06928420", "06899700",
        "06935997", "06893620", "06906150", "06918440", "06932000",
        "06894000", "06935955", "06923250", "06900640", "06909500",
        "06901500", "06907700", "06893830", "06928380", "06896000",
        "06935770", "06919500", "06893390", "06920520", "06933500",
        "06928300", "06921600", "06900800", "06906300", "06897000",
        "06918740", "06909950", "06936475", "06908000", "06935850",
        "06902995", "06893820", "06930060", "06820500", "06899500",
        "06923940", "06935755", "06904500", "06910230", "06904650",
        "06927000", "06917060", "06921720", "06918460", "06921200",
        "06905500", "06918493", "06928000", "06899900", "06917630",
        "06901205", "06923950", "06894200", "06928330", "06906800",
        "06928359", "06893940", "06927240", "06893150", "06893557",
        "06917560", "06821080", "06921070", "06935830", "06935980",
        "06902000", "06896400", "06918060", "06926290", "06921590",
        "06893578", "06930015", "06928320", "06893750", "06895000",
        "06910750", "06934000", "06935890", "06821150", "06893500",
        "06906000", "06896900",
    ]
'''
config = {
    "input_cols": [
        "latitude",
        "longitude",
        "streamflow_cfs_mean",
        "gage_height_ft_mean",
        "precipitation_mm",
        "temperature_c",
        "specific_humidity_kgkg",
    ], 
    "target": "streamflow_cfs_mean",
    "train_split": 0.8,
    "val_split": 0.9,
    "start_date": "2020-01-01",
    "end_date": "2020-12-31",
    "file_path": "flood-dataset-missouri",
    "file_name": "flood_model_missouri",
    "table": "wandb.flood_model_missouri",
    "lag_window": 3,
    "sites": [
        "06936530", "06930000",
    ]
}

pcr = processor(config) 
pcr.pull_duckdb()
(
    train_X_scaled,
    val_X_scaled,
    test_X_scaled,
    train_y_scaled,
    val_y_scaled,
    test_y_scaled,
) = pcr.return_outputs()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
import numpy as np


timesteps = config.get('lag_window', 3)
drop_col = ['latitude', 'longitude', 'site_id', 'observation_hour']

X_train = prep_x_for_tf_horiz(train_X_scaled, drop_col, timesteps) 
X_val = prep_x_for_tf_horiz(val_X_scaled, drop_col, timesteps)

y_train = prep_y_for_tf_horiz(train_y_scaled, train_X_scaled["site_id"], timesteps)
y_val = prep_y_for_tf_horiz(val_y_scaled, val_X_scaled["site_id"], timesteps)


model = Sequential([
    GRU(64, activation='tanh', return_sequences=True,),
    GRU(32, activation='tanh'),
    Dense(1, activation='linear')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

X_train shape: (536, 3, 5)
y_train shape: (536, 1)


In [4]:

history = model.fit(X_train, y_train, epochs=2, batch_size=1, validation_data=(X_val, y_val))

Epoch 1/2
536/536 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.2939 - mae: 0.2104 - val_loss: 0.0988 - val_mae: 0.1424
Epoch 2/2
536/536 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1264 - mae: 0.1485 - val_loss: 0.0509 - val_mae: 0.1280


In [5]:
print('y_val shape:', y_val.shape)
print('y_val[0]:', y_val[0])
print('val_y_scaled shape:', val_y_scaled.shape)
print('val site_ids unique:', val_X_scaled["site_id"].unique())

y_val shape: (62, 1)
y_val[0]: [-0.23173411]
val_y_scaled shape: (68, 1)
val site_ids unique: shape: (2,)
Series: 'site_id' [str]
[
	"06936530"
	"06930000"
]
